# 01 – Nettoyage des données brutes (DA-01 à DA-05)

**Objectif :** transformer `data/raw/` en `data/processed/` et documenter chaque anomalie.

Le code de nettoyage est dans `src/nettoyage.py` (réutilisable, testable, relançable en une commande).
Ce notebook sert à **montrer et expliquer** ce que fait ce code : on regarde les données *avant*, on applique les fonctions, puis on regarde *après*.

Le détail de chaque anomalie est dans `docs/journal_nettoyage.md`.

In [1]:
import os, sys
os.chdir("..")            # on se place à la racine du projet
sys.path.append("src")    # pour pouvoir importer src/nettoyage.py

import pandas as pd
from nettoyage import nettoyer_etudiants

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)


## 1. Table `etudiants`

### Avant nettoyage

In [2]:
brut = pd.read_csv("data/raw/etudiants.csv", dtype=str)
print("Lignes, colonnes :", brut.shape)
brut.head()

Lignes, colonnes : (5085, 11)


,id_etudiant,cohorte,genre,age,boursier,filiere,mention_bac,note_bac,region_origine,Distance campus (km),emploi_parallele
0,HZ38160,2025,M,19,Non,Génie civil,Assez bien,12.27,Thiès,5.7,Oui
1,HZ74513,2025,F,20,Oui,Informatique,Passable,10.91,Dakar,6.0,Non
2,HZ15685,2024,M,19,Non,Informatique,Assez bien,12.16,Thiès,19.1,Non
3,HZ69894,2025,f,19,Non,Gestion,Assez bien,12.01,Dakar,5.3,Non
4,HZ44228,2025,F,NaN,Non,Gestion,Passable,10.56,Fatick,10.7,Non


In [3]:
# Les modalités non harmonisées : la même chose écrite de plusieurs façons
for col in ["genre", "boursier", "filiere"]:
    print(brut[col].value_counts(dropna=False), "\n")

genre
M           2563
F           2115
Femme         61
Masculin      57
Homme         56
m             55
F             47
H             46
FEMININ       45
f             40
Name: count, dtype: int64 

boursier
Non    2660
Oui    1912
N        71
non      63
NON      59
0        58
no       53
O        51
oui      49
1        39
OUI      39
yes      31
Name: count, dtype: int64 

filiere
Informatique      1322
Gestion           1143
Droit              775
Communication      704
Génie civil        683
gestion             46
Gestion             38
Informatique        38
INFO                37
GESTION             37
Communication       32
informatique        30
Info                28
Genie civil         23
Droit               22
Génie Civil         21
Com                 21
communication       20
DROIT               18
droit               18
GC                  17
genie civil         12
Name: count, dtype: int64 



### Traitements appliqués
1. Colonnes renommées (`id_etudiant `, `Distance campus (km)`).
2. 60 doublons exacts supprimés, puis 25 étudiants écrits autrement (identifiant en minuscules ou avec espaces) : on garde la ligne la plus complète.
3. Modalités recodées (genre, boursier, filière, mention, région).
4. Valeurs impossibles mises à manquant (âge hors 17–35, note du bac hors 0–20, distance ≤ 0 ou 9999).
5. Manquants des variables catégorielles : modalité explicite « Inconnue » / « Inconnu ».

*Pourquoi ?* (écris ici, avec tes mots, la justification de chaque choix)

In [4]:
propre, log = nettoyer_etudiants()
log

{'lignes_avant': 5085,
 'doublons_exacts': 60,
 'doublons_id': 25,
 'age_aberrant': 14,
 'note_bac_hors_0_20': 15,
 'distance_impossible': 25,
 'manquants_mention_bac': 150,
 'manquants_region_origine': 90,
 'manquants_emploi_parallele': 200,
 'modifies_genre': 354,
 'modifies_boursier': 500,
 'modifies_filiere': 324,
 'modifies_mention_bac': 493,
 'modifies_region_origine': 289,
 'lignes_apres': 5000}

### Après nettoyage

In [5]:
print("Lignes, colonnes :", propre.shape)
print("Identifiants uniques :", propre["id_etudiant"].is_unique)
for col in ["genre", "boursier", "filiere", "mention_bac"]:
    print(propre[col].value_counts(dropna=False), "\n")

Lignes, colonnes : (5000, 11)
Identifiants uniques : True
genre
M    2730
F    2270
Name: count, dtype: int64 

boursier
Non    2923
Oui    2077
Name: count, dtype: int64 

filiere
Informatique     1438
Gestion          1236
Droit             821
Communication     763
Génie civil       742
Name: count, dtype: int64 

mention_bac
Passable      2203
Assez bien    1429
Bien           897
Très bien      321
Inconnue       150
Name: count, dtype: int64 



In [6]:
propre[["age", "note_bac", "distance_campus_km"]].describe()

,age,note_bac,distance_campus_km
count,4916.0,4985.0,4975.0
mean,20.176159,12.701262,20.290151
std,1.569123,2.023837,31.89456
min,18.0,0.0,0.9
25%,19.0,11.08,6.1
50%,20.0,12.26,9.8
75%,21.0,13.96,17.5
max,31.0,18.5,546.1


### Interprétation
*(écris ici 2 ou 3 phrases compréhensibles par un conseiller pédagogique : combien de lignes avant/après, ce qui a changé, ce qui reste manquant et pourquoi)*

---
## 2. Tables `inscriptions`, `activite_hebdo`, `tutorat`, `messages`
*(à compléter dans les tâches DA-03, DA-04 et DA-05)*